# Dataset Processing

`trn.json` → `dataset_sample.json`


In [1]:
import json
import random
import os
import html
import re
from datasets import Dataset
from huggingface_hub import login


In [2]:
INPUT_FILE = "trn.json"
OUTPUT_FILE = "dataset_sample.json"
SAMPLE_SIZE = 5000
HF_REPO = "umtaldejr/IADT-Fase-3-dataset-sample"

# Configuration for enhanced conversations
CONVERSATION_PATTERNS = {
    'basic_description': 0.2,      # 20% - Keep some original format
    'information_request': 0.15,   # 15% - "I need information about..."
    'detailed_inquiry': 0.15,      # 15% - "Can you provide details..."
    'feature_analysis': 0.15,      # 15% - "What are the key features..."
    'casual_question': 0.15,       # 15% - "What can you tell me..."
    'summary_request': 0.1,        # 10% - "Give me a summary..."
    'multi_turn': 0.1             # 10% - Multi-turn conversations
}


In [3]:
def analyze_content_type(title, content):
    """Analyze content to determine appropriate conversation patterns"""
    title_lower = title.lower()
    content_lower = content.lower()
    
    # Detect content categories
    is_product = any(word in title_lower for word in ['size', 'color', 'model', 'brand', 'pack', 'set'])
    is_book = any(word in title_lower for word in ['book', 'novel', 'guide', 'manual', 'story'])
    is_technical = any(word in content_lower for word in ['specification', 'feature', 'technical', 'system'])
    is_descriptive = len(content) > 200
    
    return {
        'is_product': is_product,
        'is_book': is_book, 
        'is_technical': is_technical,
        'is_descriptive': is_descriptive,
        'content_length': len(content)
    }

def generate_enhanced_conversations(title, content):
    """Generate diverse conversation patterns for better fine-tuning"""
    
    analysis = analyze_content_type(title, content)
    conversations = []
    
    # Template variations based on content analysis
    templates = {
        'basic_description': [
            f"Describe: {title}",
            f"What is {title}?",
            f"Describe {title} for me."
        ],
        
        'information_request': [
            f"I need information about {title}. Can you help?",
            f"Could you provide information on {title}?",
            f"I'm looking for details about {title}."
        ],
        
        'detailed_inquiry': [
            f"Can you provide details about '{title}'?",
            f"What details can you share about {title}?",
            f"I'd like to know more about {title}."
        ],
        
        'feature_analysis': [
            f"What are the key features of {title}?",
            f"What makes {title} special?",
            f"What should I know about the features of {title}?"
        ],
        
        'casual_question': [
            f"What can you tell me about '{title}'?",
            f"Tell me about {title}.",
            f"What do you know about {title}?"
        ],
        
        'summary_request': [
            f"Give me a summary of {title}.",
            f"Can you summarize {title}?",
            f"Provide a brief overview of {title}."
        ]
    }
    
    # Response variations
    response_formats = {
        'direct': content,
        'helpful': f"Here's what I can tell you about {title}:\n\n{content}",
        'detailed': f"Regarding {title}:\n\n{content}",
        'informative': f"About {title} - {content}"
    }
    
    # Generate single-turn conversations
    for pattern_type, prompts in templates.items():
        # Select appropriate response format based on content type
        if analysis['is_technical']:
            response = response_formats['detailed']
        elif analysis['is_descriptive']:
            response = response_formats['helpful']
        else:
            response = response_formats['direct']
            
        # Add variation in prompts
        selected_prompt = random.choice(prompts)
        
        conversations.append({
            'type': pattern_type,
            'conversation': {
                "conversations": [
                    {"from": "human", "value": selected_prompt},
                    {"from": "gpt", "value": response}
                ]
            }
        })
    
    # Generate multi-turn conversations for substantial content
    if analysis['content_length'] > 150:
        sentences = content.split('. ')
        if len(sentences) >= 3:
            # Split content intelligently
            mid_point = len(sentences) // 2
            first_part = '. '.join(sentences[:mid_point]).strip()
            second_part = '. '.join(sentences[mid_point:]).strip()
            
            if first_part and second_part:
                # Ensure proper sentence ending
                if not first_part.endswith('.'):
                    first_part += '.'
                if not second_part.endswith('.'):
                    second_part += '.'
                
                multi_turn_patterns = [
                    {
                        "conversations": [
                            {"from": "human", "value": f"Tell me about {title}."},
                            {"from": "gpt", "value": first_part},
                            {"from": "human", "value": "Can you provide more details?"},
                            {"from": "gpt", "value": second_part}
                        ]
                    },
                    {
                        "conversations": [
                            {"from": "human", "value": f"What is {title}?"},
                            {"from": "gpt", "value": first_part},
                            {"from": "human", "value": "What else should I know?"},
                            {"from": "gpt", "value": second_part}
                        ]
                    }
                ]
                
                for pattern in multi_turn_patterns:
                    conversations.append({
                        'type': 'multi_turn',
                        'conversation': pattern
                    })
    
    return conversations

def process():
    conversations = []
    
    with open(INPUT_FILE, 'r') as f:
        count_equal = 0
        for line in f:
            if not line.strip(): continue
            try:
                data = json.loads(line)
                title = html.unescape(data.get('title', '')).strip()
                content = html.unescape(data.get('content', '')).strip()
                
                if title == content:
                    count_equal += 1
                if title and content and title != content:
                    # Generate all possible conversation variants
                    enhanced_conversations = generate_enhanced_conversations(title, content)
                    
                    # Sample conversations based on configured distribution
                    selected_conversations = []
                    
                    # Group conversations by type
                    conversations_by_type = {}
                    for conv_data in enhanced_conversations:
                        conv_type = conv_data['type']
                        if conv_type not in conversations_by_type:
                            conversations_by_type[conv_type] = []
                        conversations_by_type[conv_type].append(conv_data['conversation'])
                    
                    # Sample according to distribution
                    for conv_type, probability in CONVERSATION_PATTERNS.items():
                        if conv_type in conversations_by_type and random.random() < probability:
                            # Select one conversation of this type
                            selected_conv = random.choice(conversations_by_type[conv_type])
                            selected_conversations.append(selected_conv)
                    
                    # Ensure at least one conversation per item
                    if not selected_conversations and enhanced_conversations:
                        fallback_conv = random.choice(enhanced_conversations)
                        selected_conversations.append(fallback_conv['conversation'])
                    
                    conversations.extend(selected_conversations)
                    
            
            except: continue
        
        print(f"❌ {count_equal} conversações com título e conteúdo iguais")
    if len(conversations) > SAMPLE_SIZE:
        conversations = random.sample(conversations, SAMPLE_SIZE)
    
    with open(OUTPUT_FILE, 'w') as f:
        json.dump(conversations, f, ensure_ascii=False, indent=2)
    
    # Calculate statistics
    single_turn = sum(1 for conv in conversations if len(conv['conversations']) == 2)
    multi_turn = sum(1 for conv in conversations if len(conv['conversations']) > 2)
    
    print(f"✅ {len(conversations)} conversações → {OUTPUT_FILE}")
    print(f"   📊 Single-turn: {single_turn}, Multi-turn: {multi_turn}")
    print(f"   📈 Improvement: {len(conversations)/5000:.1f}x more diverse conversations")

if os.path.exists(INPUT_FILE):
    process()
else:
    print(f"❌ {INPUT_FILE} não encontrado")


❌ 38840 conversações com título e conteúdo iguais
✅ 5000 conversações → dataset_sample.json
   📊 Single-turn: 4535, Multi-turn: 465
   📈 Improvement: 1.0x more diverse conversations


In [4]:
if HF_REPO != "seu-username/dataset":
    with open(OUTPUT_FILE, 'r') as f:
        data = json.load(f)
    Dataset.from_list(data).push_to_hub(HF_REPO, private=True)
    print(f"✅ {HF_REPO}")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md:   0%|          | 0.00/350 [00:00<?, ?B/s]

✅ umtaldejr/IADT-Fase-3-dataset-sample
